In [ ]:
# =========================================
# 0) 설정
# =========================================
import os, zipfile
import numpy as np
import pandas as pd

from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from scipy.sparse import csr_matrix
from sklearn.utils import murmurhash3_32

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [ ]:
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

DATA_DIR = "data"
zip_files = {
    "2507": "카드소비 데이터_202507.zip",
    "2508": "카드소비 데이터_202508.zip",
    "2509": "카드소비 데이터_202509.zip",
    "2510": "카드소비 데이터_202510.zip",
    "2511": "카드소비 데이터_202511.zip",
    "2512": "카드소비 데이터_202512.zip",
}

# --- 컬럼명(스키마 기준) ---
COL_DATE = "ta_ymd"            # YYYYMMDD
COL_CAT  = "card_tpbuz_nm_2"   # 업종 중분류명
COL_AMT  = "amt"
COL_CNT  = "cnt"
COL_HOUR = "hour"             # 시간대 코드 01~10
COL_SEX  = "sex"              # M/F
COL_AGE  = "age"              # 있으면 유지(후분석)

# --- 대용량 샘플링 파라미터(메모리에 맞춰 조절) ---
TOTAL_ROWS_EST = 75_271_815    # 대략치(있으면 좋음)
CHUNKSIZE      = 400_000

N_STATS        = 200_000       # 스케일(clip/mu/sigma) 추정
N_TRAIN        = 1_200_000     # 학습 샘플
N_SUMMARY      = 600_000       # 해석/후분석 샘플

# --- 해싱 차원(2^18=262k 추천, 메모리 여유 있으면 2^19) ---
HASH_DIM = 2**18

# --- K 탐색(너무 크면 의미가 흐려져서 보통 6~20 권장) ---
K_RANGE = [6, 8, 10, 12, 15, 20]

# =========================================
# hour 코드 매핑(스키마 그대로)
# =========================================
HOUR_CODE_TO_RANGE = {
    1: "00:00~06:59",
    2: "07:00~08:59",
    3: "09:00~10:59",
    4: "11:00~12:59",
    5: "13:00~14:59",
    6: "15:00~16:59",
    7: "17:00~18:59",
    8: "19:00~20:59",
    9: "21:00~22:59",
    10:"23:00~23:59",
}
HOUR_CODE_TO_BAND = {
    1: "새벽",
    2: "이른아침",
    3: "오전",
    4: "점심",
    5: "오후",
    6: "오후",
    7: "저녁",
    8: "저녁",
    9: "밤",
    10:"심야",
}

print("✅ 설정 완료")

In [ ]:
# =========================================
# 1) ZIP 스트리밍 제너레이터 (CSV를 chunk로 읽음)
# =========================================
def iter_zip_csv_chunks(zip_path: str, month: str, chunksize=400_000):
    with zipfile.ZipFile(zip_path) as zf:
        for filename in sorted(zf.namelist()):
            # 파일명 규칙이 다르면 여기만 수정
            city_name = filename.split("_")[-1].replace(".csv", "")
            with zf.open(filename) as f:
                it = pd.read_csv(f, chunksize=chunksize, low_memory=True)
                for chunk in it:
                    chunk["month"] = month
                    chunk["시군구"] = city_name
                    yield chunk

def iter_all_chunks(chunksize=400_000):
    for month, zipname in zip_files.items():
        zip_path = os.path.join(DATA_DIR, zipname)
        print(f"▶ 스트리밍: {month} ({zipname})")
        yield from iter_zip_csv_chunks(zip_path, month, chunksize=chunksize)

In [ ]:
# =========================================
# 2) 샘플링 유틸 (대용량 스트리밍 중 필요한 만큼만 모음)
#    - cnt 가중 샘플링(집계행이 대표하는 거래 건수 반영)
# =========================================
def sample_from_stream(total_rows_est, n_target, chunk, rng, weight_col=None):
    m = len(chunk)
    expected = n_target * (m / total_rows_est)
    n = int(expected)
    if rng.random() < (expected - n):
        n += 1
    if n <= 0:
        return None
    if n > m:
        n = m

    if weight_col and weight_col in chunk.columns:
        w = pd.to_numeric(chunk[weight_col], errors="coerce").fillna(0).clip(lower=0)
        if w.sum() > 0:
            return chunk.sample(n=n, weights=w, random_state=int(rng.integers(0, 1e9)))
    return chunk.sample(n=n, random_state=int(rng.integers(0, 1e9)))

def trim_to_target(df, n_target, weight_col=COL_CNT, random_state=RANDOM_STATE):
    if len(df) <= n_target:
        return df
    w = pd.to_numeric(df[weight_col], errors="coerce").fillna(0).clip(lower=0)
    if w.sum() > 0:
        return df.sample(n=n_target, weights=w, random_state=random_state)
    return df.sample(n=n_target, random_state=random_state)

# 필수 컬럼(스키마 기준)
need_cols = ["month", "시군구", COL_DATE, COL_CAT, COL_AMT, COL_CNT, COL_HOUR]

stats_parts, train_parts, summary_parts = [], [], []
seen_rows = 0

for chunk in iter_all_chunks(chunksize=CHUNKSIZE):
    seen_rows += len(chunk)

    missing = [c for c in need_cols if c not in chunk.columns]
    if missing:
        raise ValueError(f"필수 컬럼 누락: {missing}")

    # 필요한 컬럼만 남기기(메모리 절약)
    keep = ["month", "시군구", COL_DATE, COL_CAT, COL_AMT, COL_CNT, COL_HOUR]
    if COL_SEX in chunk.columns: keep.append(COL_SEX)
    if COL_AGE in chunk.columns: keep.append(COL_AGE)
    chunk = chunk[keep].copy()

    # 기본 정리(가벼운 필터)
    chunk[COL_CNT]  = pd.to_numeric(chunk[COL_CNT],  errors="coerce")
    chunk[COL_AMT]  = pd.to_numeric(chunk[COL_AMT],  errors="coerce")
    chunk[COL_HOUR] = pd.to_numeric(chunk[COL_HOUR], errors="coerce")
    chunk = chunk.dropna(subset=[COL_CNT, COL_AMT, COL_HOUR, COL_DATE])
    chunk = chunk[chunk[COL_CNT] > 0]
    if len(chunk) == 0:
        continue

    s1 = sample_from_stream(TOTAL_ROWS_EST, N_STATS,   chunk, rng, weight_col=COL_CNT)
    s2 = sample_from_stream(TOTAL_ROWS_EST, N_TRAIN,   chunk, rng, weight_col=COL_CNT)
    s3 = sample_from_stream(TOTAL_ROWS_EST, N_SUMMARY, chunk, rng, weight_col=COL_CNT)

    if s1 is not None: stats_parts.append(s1)
    if s2 is not None: train_parts.append(s2)
    if s3 is not None: summary_parts.append(s3)

    if seen_rows % 5_000_000 < CHUNKSIZE:
        print(f"  ... processed ~{seen_rows:,} rows")

stats_df   = pd.concat(stats_parts, ignore_index=True)   if stats_parts   else pd.DataFrame()
train_df   = pd.concat(train_parts, ignore_index=True)   if train_parts   else pd.DataFrame()
summary_df = pd.concat(summary_parts, ignore_index=True) if summary_parts else pd.DataFrame()

# 목표 크기로 트림(조금 초과해 모였을 수 있음)
if len(stats_df)   > int(N_STATS*1.05):   stats_df   = trim_to_target(stats_df,   N_STATS)
if len(train_df)   > int(N_TRAIN*1.05):   train_df   = trim_to_target(train_df,   N_TRAIN)
if len(summary_df) > int(N_SUMMARY*1.05): summary_df = trim_to_target(summary_df, N_SUMMARY)

print("✅ 샘플 완료")
print("stats_df  :", stats_df.shape)
print("train_df  :", train_df.shape)
print("summary_df:", summary_df.shape)

In [ ]:
# =========================================
# 3) 파생변수 생성(스키마 반영) + robust clip/mu/sigma 추정
#    - hour: 01~10 코드 -> range/band 라벨
#    - ta_ymd -> 요일/주말
#    - ticket_bucket 추가(분리력↑)
# =========================================
def add_derived(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 타입/결측 정리
    df[COL_CNT]  = pd.to_numeric(df[COL_CNT], errors="coerce")
    df[COL_AMT]  = pd.to_numeric(df[COL_AMT], errors="coerce")
    df[COL_HOUR] = pd.to_numeric(df[COL_HOUR], errors="coerce")
    df = df.dropna(subset=[COL_CNT, COL_AMT, COL_HOUR, COL_DATE])
    df = df[df[COL_CNT] > 0].copy()

    # ticket/log_ticket
    df["ticket"] = df[COL_AMT] / df[COL_CNT]
    df = df[df["ticket"] > 0].copy()
    df["log_ticket"] = np.log1p(df["ticket"])

    # 날짜 -> 요일/주말 (YYYYMMDD)
    s = df[COL_DATE].astype(str).str.zfill(8)
    dt = pd.to_datetime(s, format="%Y%m%d", errors="coerce")
    df = df[dt.notna()].copy()
    dt = dt.loc[df.index]

    df["dow"] = dt.dt.dayofweek  # 월0~일6
    df["is_weekend"] = np.where(df["dow"] >= 5, "주말", "평일")

    # hour 코드 -> 라벨
    h = df[COL_HOUR].astype(int)
    df["hour_code"]  = h.map(lambda x: f"{x:02d}")
    df["hour_range"] = h.map(HOUR_CODE_TO_RANGE).fillna("UNK")
    df["hour_band"]  = h.map(HOUR_CODE_TO_BAND).fillna("UNK")

    # 업종 결측
    df[COL_CAT] = df[COL_CAT].astype(str).fillna("기타")

    # 금액 구간
    bins = [-1, 3000, 10000, 30000, 100000, np.inf]
    labels = ["~3천", "3천~1만", "1만~3만", "3만~10만", "10만~"]
    df["ticket_bucket"] = pd.cut(df["ticket"], bins=bins, labels=labels).astype(str)

    return df

def weighted_quantile(values, quantiles, sample_weight):
    values = np.asarray(values, dtype=float)
    quantiles = np.asarray(quantiles, dtype=float)
    sample_weight = np.asarray(sample_weight, dtype=float)

    mask = np.isfinite(values) & np.isfinite(sample_weight) & (sample_weight >= 0)
    values = values[mask]
    sample_weight = sample_weight[mask]
    if len(values) == 0 or sample_weight.sum() == 0:
        return np.array([np.nan]*len(quantiles))

    sorter = np.argsort(values)
    values = values[sorter]
    sample_weight = sample_weight[sorter]
    cum_w = np.cumsum(sample_weight)
    cum_w = cum_w / cum_w[-1]
    return np.interp(quantiles, cum_w, values)

stats_df = add_derived(stats_df)

w = stats_df[COL_CNT].to_numpy(dtype=float)
x = stats_df["log_ticket"].to_numpy(dtype=float)

clip_lo, clip_hi = weighted_quantile(x, [0.005, 0.995], w)  # 0.5%~99.5%
x_clip = np.clip(x, clip_lo, clip_hi)

mu = np.sum(w * x_clip) / (w.sum() + 1e-12)
var = np.sum(w * (x_clip - mu)**2) / (w.sum() + 1e-12)
sigma = float(np.sqrt(var)) if var > 1e-12 else 1.0

print("✅ log_ticket clip:", clip_lo, clip_hi)
print("✅ log_ticket mean/std:", mu, sigma)

In [ ]:
# =========================================
# 4) 해싱 기반 sparse 피처 생성
#    - ✅ 소비행동 피처만 사용 (업종/시간대코드/시간대밴드/요일/주말/금액구간 + log_ticket)
#    - ❌ sex/age/시군구/month는 df에 남겨두되 feature에 넣지 않음
# =========================================
def make_hashed_X(df, hash_dim=HASH_DIM, mu=0.0, sigma=1.0, clip_lo=None, clip_hi=None):
    df = add_derived(df)

    # numeric 표준화
    z = df["log_ticket"].to_numpy(dtype=float)
    if clip_lo is not None and clip_hi is not None:
        z = np.clip(z, clip_lo, clip_hi)
    z = (z - mu) / sigma

    # categorical tokens
    cat  = df[COL_CAT].astype(str).to_numpy()
    hc   = df["hour_code"].astype(str).to_numpy()
    hb   = df["hour_band"].astype(str).to_numpy()
    dow  = df["dow"].astype(int).astype(str).to_numpy()
    wk   = df["is_weekend"].astype(str).to_numpy()
    tb   = df["ticket_bucket"].astype(str).to_numpy()

    n = len(df)
    # 6 tokens + 1 numeric = 7 nonzeros/row
    rows = np.repeat(np.arange(n), 7)

    def hidx(prefix, arr, seed):
        return np.fromiter((murmurhash3_32(prefix + v, seed=seed) % hash_dim for v in arr),
                           dtype=np.int64, count=n)

    idx1 = hidx("cat=", cat, seed=0)
    idx2 = hidx("hc=",  hc,  seed=1)
    idx3 = hidx("hb=",  hb,  seed=2)
    idx4 = hidx("dow=", dow, seed=3)
    idx5 = hidx("wk=",  wk,  seed=4)
    idx6 = hidx("tb=",  tb,  seed=5)
    idx7 = np.full(n, hash_dim, dtype=np.int64)  # numeric col

    cols = np.concatenate([idx1, idx2, idx3, idx4, idx5, idx6, idx7])
    data = np.concatenate([
        np.ones(n, dtype=np.float32),
        np.ones(n, dtype=np.float32),
        np.ones(n, dtype=np.float32),
        np.ones(n, dtype=np.float32),
        np.ones(n, dtype=np.float32),
        np.ones(n, dtype=np.float32),
        z.astype(np.float32)
    ])

    X = csr_matrix((data, (rows, cols)), shape=(n, hash_dim + 1))
    return X, df

X_train, train_df2 = make_hashed_X(train_df, hash_dim=HASH_DIM, mu=mu, sigma=sigma, clip_lo=clip_lo, clip_hi=clip_hi)
w_train = train_df2[COL_CNT].to_numpy(dtype=float)

print("✅ X_train:", X_train.shape, "nnz:", X_train.nnz)

In [ ]:
# =========================================
# 5) K 탐색(가볍게) + 최종 MiniBatchKMeans 학습
# =========================================
eval_n = min(120_000, X_train.shape[0])
idx = rng.choice(X_train.shape[0], size=eval_n, replace=False)
X_eval = X_train[idx]
w_eval = w_train[idx]

sil_scores = []
for k in K_RANGE:
    km = MiniBatchKMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        batch_size=8192,
        n_init=5,
        init="k-means++"
    )
    km.fit(X_eval, sample_weight=w_eval)
    labels = km.predict(X_eval)
    sil = silhouette_score(X_eval, labels, sample_size=min(5000, len(labels)), random_state=RANDOM_STATE)
    sil_scores.append(sil)
    print(f"k={k:2d} | silhouette={sil:.4f}")

best_k = K_RANGE[int(np.argmax(sil_scores))]
print("\n★ best_k =", best_k)

plt.figure(figsize=(10,4))
plt.plot(K_RANGE, sil_scores, marker="o")
plt.axvline(best_k, linestyle="--")
plt.title("Silhouette (hashed features sample)")
plt.xlabel("K")
plt.ylabel("silhouette")
plt.grid(linestyle="--", alpha=0.5)
plt.show()

km_final = MiniBatchKMeans(
    n_clusters=best_k,
    random_state=RANDOM_STATE,
    batch_size=8192,
    n_init=10,
    init="k-means++"
)
km_final.fit(X_train, sample_weight=w_train)
print("✅ 최종 학습 완료. n_clusters =", best_k)

In [ ]:
# =========================================
# 6) summary 샘플에 클러스터 부여 + 행동 요약/후분석
# =========================================
X_sum, summary_df2 = make_hashed_X(summary_df, hash_dim=HASH_DIM, mu=mu, sigma=sigma, clip_lo=clip_lo, clip_hi=clip_hi)
summary_df2["cluster"] = km_final.predict(X_sum)

print("✅ summary labeled:", summary_df2.shape)
summary_df2[["cluster", "month", "시군구", COL_DATE, "hour_code", "hour_range", "dow", "ticket_bucket"]].head()

In [ ]:
# =========================================
# 7) 해석(핵심): TOP 비중이 아니라 "lift(과대표)"로 봐야 함
#    - (클러스터 내 비중) / (전체 비중) = lift
# =========================================
def wavg(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    return float(np.average(x, weights=w)) if w.sum() > 0 else float(np.mean(x))

# (A) 클러스터 핵심 행동지표
core = (summary_df2.groupby("cluster")
        .apply(lambda d: pd.Series({
            "거래비중_cnt": d[COL_CNT].sum(),
            "평균티켓(가중)": wavg(d["ticket"], d[COL_CNT]),
            "주말비율(가중)": wavg((d["is_weekend"]=="주말").astype(int), d[COL_CNT]),
        }))
        .reset_index())

core["거래비중(%)"] = core["거래비중_cnt"] / core["거래비중_cnt"].sum() * 100
core = core.sort_values("거래비중_cnt", ascending=False)
display(core)

def lift_table(df, feature_col, top_n=10, stop=set()):
    g_all = df.groupby(feature_col)[COL_CNT].sum()
    p_all = g_all / g_all.sum()

    rows = []
    for c, d in df.groupby("cluster"):
        g = d.groupby(feature_col)[COL_CNT].sum()
        p = g / g.sum()
        lift = (p / p_all).replace([np.inf, -np.inf], np.nan).dropna()
        lift = lift[~lift.index.isin(stop)]
        top = lift.sort_values(ascending=False).head(top_n)

        for k, v in top.items():
            rows.append({
                "cluster": int(c),
                feature_col: k,
                "lift": float(v),
                "cluster_share(%)": float(p.get(k, 0) * 100),
                "global_share(%)": float(p_all.get(k, 0) * 100),
            })
    return pd.DataFrame(rows).sort_values(["cluster","lift"], ascending=[True, False])

# (B) 업종/시간/요일/티켓구간 lift
STOP_CAT = {"종합소매점", "기타"}  # 너무 흔한 항목은 해석에서 제외(원하면 수정)

cat_lift   = lift_table(summary_df2, COL_CAT,        top_n=8, stop=STOP_CAT)
hour_lift  = lift_table(summary_df2, "hour_range",   top_n=5)
dow_lift   = lift_table(summary_df2, "dow",          top_n=3)
buck_lift  = lift_table(summary_df2, "ticket_bucket",top_n=5)

display(cat_lift.head(30))
display(hour_lift.head(30))
display(buck_lift.head(30))

In [ ]:
# =========================================
# 8) 자동 라벨(사람이 읽는 클러스터 이름)
# =========================================
def make_labels(core_df, cat_lift_df, hour_lift_df, buck_lift_df):
    labels = {}
    for _, r in core_df.iterrows():
        c = int(r["cluster"])

        top_cat = (cat_lift_df[cat_lift_df["cluster"]==c]
                   .head(2)[COL_CAT].tolist())
        top_hour = (hour_lift_df[hour_lift_df["cluster"]==c]
                    .head(1)["hour_range"].tolist())
        top_buck = (buck_lift_df[buck_lift_df["cluster"]==c]
                    .head(1)["ticket_bucket"].tolist())

        p50_guess = r["평균티켓(가중)"]
        lvl = "소액" if p50_guess < 5000 else "중소액" if p50_guess < 20000 else "중고액" if p50_guess < 80000 else "고액"
        wk  = "주말형" if r["주말비율(가중)"] >= 0.45 else "평일형"

        labels[c] = f"[{lvl}/{wk}] " \
                    f"{(top_hour[0] if top_hour else '')} · {(top_buck[0] if top_buck else '')} | " \
                    f"{' / '.join(top_cat) if top_cat else ''}"
    return labels

label_map = make_labels(core, cat_lift, hour_lift, buck_lift)
label_df = pd.DataFrame({"cluster": sorted(label_map), "label": [label_map[k] for k in sorted(label_map)]})
display(label_df)

summary_df2["cluster_label"] = summary_df2["cluster"].map(label_map)
summary_df2[["cluster","cluster_label","hour_range","ticket_bucket",COL_CAT]].head()

In [ ]:
# =========================================
# 8) 자동 라벨(사람이 읽는 클러스터 이름)
# =========================================
def make_labels(core_df, cat_lift_df, hour_lift_df, buck_lift_df):
    labels = {}
    for _, r in core_df.iterrows():
        c = int(r["cluster"])

        top_cat = (cat_lift_df[cat_lift_df["cluster"]==c]
                   .head(2)[COL_CAT].tolist())
        top_hour = (hour_lift_df[hour_lift_df["cluster"]==c]
                    .head(1)["hour_range"].tolist())
        top_buck = (buck_lift_df[buck_lift_df["cluster"]==c]
                    .head(1)["ticket_bucket"].tolist())

        p50_guess = r["평균티켓(가중)"]
        lvl = "소액" if p50_guess < 5000 else "중소액" if p50_guess < 20000 else "중고액" if p50_guess < 80000 else "고액"
        wk  = "주말형" if r["주말비율(가중)"] >= 0.45 else "평일형"

        labels[c] = f"[{lvl}/{wk}] " \
                    f"{(top_hour[0] if top_hour else '')} · {(top_buck[0] if top_buck else '')} | " \
                    f"{' / '.join(top_cat) if top_cat else ''}"
    return labels

label_map = make_labels(core, cat_lift, hour_lift, buck_lift)
label_df = pd.DataFrame({"cluster": sorted(label_map), "label": [label_map[k] for k in sorted(label_map)]})
display(label_df)

summary_df2["cluster_label"] = summary_df2["cluster"].map(label_map)
summary_df2[["cluster","cluster_label","hour_range","ticket_bucket",COL_CAT]].head()

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# ---- 컬럼명(너 환경에 맞게) ----
WCOL   = COL_CNT              # 'cnt'
AMTCOL = COL_AMT              # 'amt'
CATCOL = COL_CAT              # 'card_tpbuz_nm_2'

REQ = ["cluster", WCOL, AMTCOL, "ticket", "is_weekend", "hour_range", "dow", "ticket_bucket", CATCOL]
miss = [c for c in REQ if c not in summary_df2.columns]
if miss:
    raise ValueError(f"summary_df2에 필요한 컬럼이 없습니다: {miss}")

def wavg(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    return float(np.average(x, weights=w)) if w.sum() > 0 else float(np.mean(x))

def weighted_quantile(values, quantiles, sample_weight):
    values = np.asarray(values, dtype=float)
    quantiles = np.asarray(quantiles, dtype=float)
    sample_weight = np.asarray(sample_weight, dtype=float)

    mask = np.isfinite(values) & np.isfinite(sample_weight) & (sample_weight >= 0)
    values = values[mask]
    sample_weight = sample_weight[mask]
    if len(values) == 0 or sample_weight.sum() == 0:
        return np.array([np.nan]*len(quantiles))

    sorter = np.argsort(values)
    values = values[sorter]
    sample_weight = sample_weight[sorter]
    cw = np.cumsum(sample_weight)
    cw /= cw[-1]
    return np.interp(quantiles, cw, values)

def cluster_stats(df):
    rows=[]
    total_cnt = df[WCOL].sum()
    total_amt = df[AMTCOL].sum()

    for c, d in df.groupby("cluster"):
        w = d[WCOL].to_numpy()
        q = weighted_quantile(d["ticket"].to_numpy(), [0.1,0.25,0.5,0.75,0.9], w)
        rows.append({
            "cluster": int(c),
            "거래건수(가중)": float(w.sum()),
            "거래비중(%)": float(w.sum()/total_cnt*100) if total_cnt>0 else np.nan,
            "결제금액(가중)": float(d[AMTCOL].sum()),
            "금액비중(%)": float(d[AMTCOL].sum()/total_amt*100) if total_amt>0 else np.nan,
            "평균티켓(가중)": wavg(d["ticket"], w),
            "티켓_p10": q[0], "티켓_p25": q[1], "티켓_p50": q[2], "티켓_p75": q[3], "티켓_p90": q[4],
            "주말비율(가중)": wavg((d["is_weekend"]=="주말").astype(int), w),
        })
    return pd.DataFrame(rows).sort_values("거래건수(가중)", ascending=False)

core_stats = cluster_stats(summary_df2)
display(core_stats)

In [ ]:
def lift_table(df, feature_col, top_n=10, stop=set()):
    # 전체 분포
    g_all = df.groupby(feature_col)[WCOL].sum()
    p_all = g_all / g_all.sum()

    out = []
    for c, d in df.groupby("cluster"):
        g = d.groupby(feature_col)[WCOL].sum()
        p = g / g.sum()

        lift = (p / p_all).replace([np.inf, -np.inf], np.nan).dropna()
        lift = lift[~lift.index.isin(stop)]
        top = lift.sort_values(ascending=False).head(top_n)

        for k, v in top.items():
            out.append({
                "cluster": int(c),
                "feature": feature_col,
                "value": k,
                "lift": float(v),
                "cluster_share(%)": float(p.get(k, 0) * 100),
                "global_share(%)": float(p_all.get(k, 0) * 100),
            })
    return pd.DataFrame(out).sort_values(["cluster","lift"], ascending=[True, False])

def top_combinations(df, cluster_id, cols, top_n=15):
    d = df[df["cluster"]==cluster_id]
    g = (d.groupby(cols)[WCOL].sum()
           .reset_index()
           .sort_values(WCOL, ascending=False))
    g["share_in_cluster(%)"] = g[WCOL] / g[WCOL].sum() * 100
    return g.head(top_n)

# 해석 방해하는 베이스레이트 stopword (원하면 더 추가)
STOP_CAT = {"종합소매점", "기타"}  # 너무 흔하면 인터넷쇼핑도 추가 고려

cat_lift  = lift_table(summary_df2, CATCOL,         top_n=10, stop=STOP_CAT)
time_lift = lift_table(summary_df2, "hour_range",   top_n=5,  stop=set())
dow_lift  = lift_table(summary_df2, "dow",          top_n=3,  stop=set())
buck_lift = lift_table(summary_df2, "ticket_bucket",top_n=5,  stop=set())

display(cat_lift.head(30))
display(time_lift.head(20))
display(buck_lift.head(20))

In [ ]:
import numpy as np
import pandas as pd
import re
from IPython.display import display, HTML

# -----------------------------
# 0) 컬럼 자동 탐지
# -----------------------------
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

WCOL   = pick_col(summary_df2, ["cnt", "count", "CNT"])
AMTCOL = pick_col(summary_df2, ["amt", "amount", "AMT"])
CATCOL = pick_col(summary_df2, ["card_tpbuz_nm_2", "card_tpbuz_nm_1", "업종", "category"])

if WCOL is None or AMTCOL is None or CATCOL is None:
    raise ValueError(f"필수 컬럼 못 찾음: WCOL={WCOL}, AMTCOL={AMTCOL}, CATCOL={CATCOL}. "
                     f"현재 columns={list(summary_df2.columns)[:30]}...")

df = summary_df2.copy()

# -----------------------------
# 1) 없으면 파생 컬럼 생성(방어)
# -----------------------------
if "ticket" not in df.columns:
    df["ticket"] = pd.to_numeric(df[AMTCOL], errors="coerce") / pd.to_numeric(df[WCOL], errors="coerce")

if "ticket_bucket" not in df.columns:
    bins = [-1, 3000, 10000, 30000, 100000, np.inf]
    labels = ["~3천", "3천~1만", "1만~3만", "3만~10만", "10만~"]
    df["ticket_bucket"] = pd.cut(df["ticket"], bins=bins, labels=labels).astype(str)

if "dow" not in df.columns:
    # dow 없으면 주말만이라도 있으면 OK
    pass

if "is_weekend" not in df.columns:
    if "dow" in df.columns:
        df["is_weekend"] = np.where(pd.to_numeric(df["dow"], errors="coerce") >= 5, "주말", "평일")
    else:
        # 최후 fallback
        df["is_weekend"] = "UNK"

# hour_range 없으면 hour 코드(1~10) 가정하고 매핑
if "hour_range" not in df.columns:
    if "hour" in df.columns:
        HOUR_CODE_TO_RANGE = {
            1:"00:00~06:59",2:"07:00~08:59",3:"09:00~10:59",4:"11:00~12:59",5:"13:00~14:59",
            6:"15:00~16:59",7:"17:00~18:59",8:"19:00~20:59",9:"21:00~22:59",10:"23:00~23:59"
        }
        h = pd.to_numeric(df["hour"], errors="coerce").fillna(-1).astype(int)
        df["hour_range"] = h.map(HOUR_CODE_TO_RANGE).fillna("UNK")
    else:
        df["hour_range"] = "UNK"

# cluster 필수
if "cluster" not in df.columns:
    raise ValueError("df에 cluster 컬럼이 없습니다.")

# 숫자형 정리
df[WCOL]   = pd.to_numeric(df[WCOL], errors="coerce").fillna(0)
df[AMTCOL] = pd.to_numeric(df[AMTCOL], errors="coerce").fillna(0)
df["ticket"] = pd.to_numeric(df["ticket"], errors="coerce").fillna(0)

# -----------------------------
# 2) 가중 통계/분위수
# -----------------------------
def wavg(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    return float(np.average(x, weights=w)) if w.sum() > 0 else float(np.mean(x))

def weighted_quantile(values, quantiles, sample_weight):
    values = np.asarray(values, dtype=float)
    quantiles = np.asarray(quantiles, dtype=float)
    sample_weight = np.asarray(sample_weight, dtype=float)

    mask = np.isfinite(values) & np.isfinite(sample_weight) & (sample_weight >= 0)
    values = values[mask]
    sample_weight = sample_weight[mask]
    if len(values) == 0 or sample_weight.sum() == 0:
        return np.array([np.nan]*len(quantiles))

    sorter = np.argsort(values)
    values = values[sorter]
    sample_weight = sample_weight[sorter]
    cw = np.cumsum(sample_weight)
    cw /= cw[-1]
    return np.interp(quantiles, cw, values)

total_cnt = df[WCOL].sum()
total_amt = df[AMTCOL].sum()

core_rows = []
for c, d in df.groupby("cluster"):
    w = d[WCOL].to_numpy()
    q = weighted_quantile(d["ticket"].to_numpy(), [0.1,0.25,0.5,0.75,0.9], w)
    core_rows.append({
        "cluster": int(c),
        "거래비중(%)": float(w.sum()/total_cnt*100) if total_cnt>0 else np.nan,
        "금액비중(%)": float(d[AMTCOL].sum()/total_amt*100) if total_amt>0 else np.nan,
        "평균티켓(가중)": wavg(d["ticket"], w),
        "티켓_p50": q[2],
        "티켓_p90": q[4],
        "주말비율(가중)": wavg((d["is_weekend"]=="주말").astype(int), w)
    })

core = pd.DataFrame(core_rows).sort_values("거래비중(%)", ascending=False)
display(core)

# -----------------------------
# 3) lift 계산(과대표)
# -----------------------------
def lift_table(df, feature_col, top_n=8, stop=set()):
    g_all = df.groupby(feature_col)[WCOL].sum()
    p_all = g_all / g_all.sum()

    rows=[]
    for c, d in df.groupby("cluster"):
        g = d.groupby(feature_col)[WCOL].sum()
        p = g / g.sum()

        lift = (p / p_all).replace([np.inf, -np.inf], np.nan).dropna()
        lift = lift[~lift.index.isin(stop)]
        top = lift.sort_values(ascending=False).head(top_n)

        for k, v in top.items():
            rows.append({
                "cluster": int(c),
                "value": k,
                "lift": float(v),
                "cluster_share(%)": float(p.get(k,0)*100),
                "global_share(%)": float(p_all.get(k,0)*100),
            })
    return pd.DataFrame(rows).sort_values(["cluster","lift"], ascending=[True, False])

STOP_CAT = {"종합소매점", "기타"}  # 필요하면 추가
cat_lift  = lift_table(df, CATCOL, top_n=6, stop=STOP_CAT)
time_lift = lift_table(df, "hour_range", top_n=3, stop=set())
buck_lift = lift_table(df, "ticket_bucket", top_n=3, stop=set())

# -----------------------------
# 4) 대표 조합 TOP(업종×시간×금액)
# -----------------------------
def top_combos(df, cid, top_n=10):
    d = df[df["cluster"]==cid]
    g = (d.groupby([CATCOL, "hour_range", "ticket_bucket"])[WCOL].sum()
           .reset_index().sort_values(WCOL, ascending=False))
    g["share_in_cluster(%)"] = g[WCOL] / g[WCOL].sum() * 100
    return g.head(top_n)

def lift_text(lift_df, cid, top_n=3):
    sub = lift_df[lift_df["cluster"]==cid].head(top_n)
    return " · ".join([f"{r['value']}({r['lift']:.2f}×)" for _, r in sub.iterrows()]) if len(sub) else "-"

# -----------------------------
# 5) 카드 렌더(중요: display(HTML(...))로 강제 렌더)
# -----------------------------
cards=[]
for cid in sorted(df["cluster"].unique()):
    r = core[core["cluster"]==cid].iloc[0]
    lvl = "소액" if r["티켓_p50"] < 5000 else "중소액" if r["티켓_p50"] < 20000 else "중고액" if r["티켓_p50"] < 80000 else "고액"
    wk  = "주말형" if r["주말비율(가중)"] >= 0.45 else "평일형"

    ccat  = lift_text(cat_lift, cid, 3)
    ctime = lift_text(time_lift, cid, 2)
    cbuck = lift_text(buck_lift, cid, 2)

    combo = top_combos(df, cid, 8)
    combo_html = "<br>".join([
        f"{row[CATCOL]} / {row['hour_range']} / {row['ticket_bucket']} ({row['share_in_cluster(%)']:.1f}%)"
        for _, row in combo.iterrows()
    ]) if len(combo) else "-"

    cards.append(f"""
    <div style="border:1px solid #333;border-radius:14px;padding:14px;background:#111;">
      <div style="font-size:16px;font-weight:800;margin-bottom:8px;">Cluster {cid} · {lvl}/{wk}</div>
      <div style="display:flex;gap:8px;flex-wrap:wrap;margin-bottom:10px;font-size:13px;">
        <span style="border:1px solid #444;border-radius:999px;padding:3px 10px;">거래 {r['거래비중(%)']:.1f}%</span>
        <span style="border:1px solid #444;border-radius:999px;padding:3px 10px;">금액 {r['금액비중(%)']:.1f}%</span>
        <span style="border:1px solid #444;border-radius:999px;padding:3px 10px;">평균티켓 {r['평균티켓(가중)']:,.0f}</span>
        <span style="border:1px solid #444;border-radius:999px;padding:3px 10px;">p50 {r['티켓_p50']:,.0f}</span>
        <span style="border:1px solid #444;border-radius:999px;padding:3px 10px;">p90 {r['티켓_p90']:,.0f}</span>
      </div>
      <div style="font-size:13px;line-height:1.45;margin-bottom:10px;">
        <b>업종 lift</b>: {ccat}<br>
        <b>시간대 lift</b>: {ctime}<br>
        <b>금액구간 lift</b>: {cbuck}
      </div>
      <div style="font-size:13px;line-height:1.45;">
        <b>대표 조합 TOP8 (업종/시간/금액)</b><br>{combo_html}
      </div>
    </div>
    """)

html = "<div style='display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px;'>" + "".join(cards) + "</div>"
display(HTML(html))

In [ ]:
from IPython.display import display, HTML
import html

# lift 텍스트( "A(2.54×) · B(2.41×) ..." ) -> pill 리스트로
def split_lift_pills(s):
    if not s or s == "-":
        return "<span class='pill muted'>-</span>"
    parts = [p.strip() for p in str(s).split("·")]
    pills = []
    for p in parts:
        if p:
            pills.append(f"<span class='pill'>{html.escape(p)}</span>")
    return "".join(pills) if pills else "<span class='pill muted'>-</span>"

# combo df -> li list
def combo_df_to_li(combo_df, cat_col):
    if combo_df is None or len(combo_df) == 0:
        return "<li class='muted'>-</li>"
    li=[]
    for _, r in combo_df.iterrows():
        li.append(
            f"<li><span class='mono'>{html.escape(str(r[cat_col]))}</span>"
            f"<span class='sep'>·</span>{html.escape(str(r['hour_range']))}"
            f"<span class='sep'>·</span>{html.escape(str(r['ticket_bucket']))}"
            f"<span class='pct'>{r['share_in_cluster(%)']:.1f}%</span></li>"
        )
    return "".join(li)

# --------- CSS(스샷보다 훨씬 깔끔) ----------
css = """
<style>
:root{
  --bg:#0b0d10; --card:#101318; --card2:#0f1217;
  --border:#242a33; --text:#e9eef7; --muted:#aab3c2;
  --pill:#151a22; --pillbd:#2b3340;
}
body{background:var(--bg); color:var(--text);}
.grid{
  display:grid;
  grid-template-columns:repeat(auto-fit,minmax(420px,1fr));
  gap:14px;
}
.card{
  background:linear-gradient(180deg,var(--card),var(--card2));
  border:1px solid var(--border);
  border-radius:16px;
  padding:16px;
  box-shadow:0 8px 24px rgba(0,0,0,.35);
}
.head{
  display:flex; align-items:flex-start; justify-content:space-between; gap:12px;
  margin-bottom:10px;
}
.h1{
  font-size:18px; font-weight:800; letter-spacing:-0.2px; margin:0;
}
.sub{
  margin-top:4px; font-size:13px; color:var(--muted);
}
.badge{
  font-size:12px; padding:4px 10px; border-radius:999px;
  border:1px solid var(--border); background:#0c0f14; color:var(--muted);
  white-space:nowrap;
}
.chips{display:flex; flex-wrap:wrap; gap:8px; margin:10px 0 12px;}
.chip{
  border:1px solid var(--border); background:#0c0f14;
  border-radius:999px; padding:4px 10px; font-size:12px; color:var(--text);
}
.chip b{font-weight:800}
.section{margin-top:12px;}
.stitle{
  font-size:13px; font-weight:800; margin:0 0 8px 0;
}
.pills{display:flex; flex-wrap:wrap; gap:8px;}
.pill{
  border:1px solid var(--pillbd); background:var(--pill);
  border-radius:999px; padding:4px 10px; font-size:12px;
}
.pill.muted{color:var(--muted)}
.kv{
  display:grid;
  grid-template-columns:110px 1fr;
  gap:8px 10px;
  font-size:13px;
}
.k{color:var(--muted)}
.v{color:var(--text)}
.hr{height:1px;background:var(--border);margin:12px 0;}
.details{
  border:1px solid var(--border);
  border-radius:12px;
  padding:10px 12px;
  background:#0c0f14;
}
details summary{
  cursor:pointer;
  font-weight:800;
  list-style:none;
  outline:none;
  font-size:13px;
}
details summary::-webkit-details-marker{display:none;}
.list{
  margin:10px 0 0 0;
  padding:0 0 0 18px;
  max-height:180px; overflow:auto;
}
.list li{margin:6px 0; line-height:1.35}
.mono{font-family:ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace;}
.sep{color:var(--muted); margin:0 6px;}
.pct{color:var(--muted); margin-left:8px; font-size:12px;}
.muted{color:var(--muted)}
</style>
"""

# --------- 카드 생성 ----------
cards=[]
for cid in sorted(df["cluster"].unique()):
    r = core[core["cluster"]==cid].iloc[0]

    lvl = "소액" if r["티켓_p50"] < 5000 else "중소액" if r["티켓_p50"] < 20000 else "중고액" if r["티켓_p50"] < 80000 else "고액"
    wk  = "주말형" if r["주말비율(가중)"] >= 0.45 else "평일형"

    # 기존에 만들던 lift 텍스트를 그대로 사용 (없으면 아래 lift_text 함수로 생성해도 됨)
    ccat  = lift_text(cat_lift, cid, 3)
    ctime = lift_text(time_lift, cid, 2)
    cbuck = lift_text(buck_lift, cid, 2)

    # 대표 조합 DF
    combo = top_combos(df, cid, 12)
    combo_li = combo_df_to_li(combo, CATCOL)

    cards.append(f"""
    <div class="card">
      <div class="head">
        <div>
          <p class="h1">Cluster {cid} · {lvl}/{wk}</p>
          <div class="sub">거래 패턴(업종×시간×금액)의 과대표(lift) 기반 요약</div>
        </div>
        <div class="badge">K={len(sorted(df["cluster"].unique()))}</div>
      </div>

      <div class="chips">
        <span class="chip">거래 <b>{r['거래비중(%)']:.1f}%</b></span>
        <span class="chip">금액 <b>{r['금액비중(%)']:.1f}%</b></span>
        <span class="chip">평균티켓 <b>{r['평균티켓(가중)']:,.0f}</b></span>
        <span class="chip">p50 <b>{r['티켓_p50']:,.0f}</b></span>
        <span class="chip">p90 <b>{r['티켓_p90']:,.0f}</b></span>
        <span class="chip">주말 <b>{r['주말비율(가중)']:.0%}</b></span>
      </div>

      <div class="kv">
        <div class="k">업종 lift</div><div class="v"><div class="pills">{split_lift_pills(ccat)}</div></div>
        <div class="k">시간대 lift</div><div class="v"><div class="pills">{split_lift_pills(ctime)}</div></div>
        <div class="k">금액구간 lift</div><div class="v"><div class="pills">{split_lift_pills(cbuck)}</div></div>
      </div>

      <div class="hr"></div>

      <div class="details">
        <details>
          <summary>대표 조합 TOP12 (업종 · 시간 · 금액구간)</summary>
          <ul class="list">{combo_li}</ul>
        </details>
      </div>
    </div>
    """)

html_out = css + "<div class='grid'>" + "".join(cards) + "</div>"
display(HTML(html_out))